# 03b: Filtering and clustering (batch)

**Step 2 of 3 of the analysis of one sample.** Runs unattended right after 03a through `run_batch.py`. It reads the corrected per-cell table from 03a and follows the SPACEc preprocessing tutorial ([03.1](https://github.com/yuqiyuqitan/SPACEc/blob/master/notebooks/tutorials/03.1_preprocessing.ipynb)) and clustering tutorial ([04](https://github.com/yuqiyuqitan/SPACEc/blob/master/notebooks/tutorials/04_clustering.ipynb)), without spillover compensation:

**cells in the GeoJSON region → DAPI/area filter (lowest 1 %) → z-score → noise removal (top 1 %) → AnnData → Leiden clustering → UMAP, dotplot, matrixplot, cluster summary and cluster maps**

Then open **03c_annotate** to subcluster and name the clusters.

| Output (in `<OUTPUT_ROOT>/<sample>/`) | |
|---|---|
| `figures/<PLOT_TAG>_4-0*.png`, `figures/<PLOT_TAG>_4-05_cluster_summary.csv` | everything you need to annotate |
| `<sample>_adata_clustered.h5ad` | clustered data, the input for 03c |
| `<sample>_cluster_config.json` | clustering settings and region area, read by 03c |

---
# Part 0: Setup
## 0.1 Sample
Filled in automatically by `run_batch.py`. To run one sample by hand, set `SAMPLE` and `OUTPUT_ROOT`. `REGION_GEOJSON` empty = the outline that 03a used.

In [ ]:
SAMPLE         = 'ss1'                                # <-- EDIT
OUTPUT_ROOT    = 'path/to/output'      # <-- EDIT same as in 03a
REGION_GEOJSON = ''                                   # empty: the outline used in 03a
PIXEL_SIZE_UM  = 0.5                                  # <-- EDIT µm per pixel (scanner metadata)

## 0.2 Load the sample settings and libraries
Reads the settings that 03a saved for this sample: condition, marker panel (including sample-specific markers such as CD206), image size and file names.

In [ ]:
import os, json, warnings, logging
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.path import Path as MplPath
from scipy.spatial import cKDTree
from skimage.draw import polygon as draw_polygon

warnings.filterwarnings('ignore')
plt.rc('axes', grid=False)

output_dir = os.path.join(OUTPUT_ROOT, SAMPLE) + os.sep
OUT = output_dir + SAMPLE + '_'
cfg = json.load(open(OUT + 'run_config.json'))
globals().update({k: cfg[k] for k in ('CONDITION', 'REGION', 'PLOT_TAG', 'H', 'W', 'BG_DS', 'NUCLEUS_CHANNEL',
                                      'marker_channels', 'LINEAGE_MARKERS', 'STATE_MARKERS', 'NUCLEAR_MARKERS',
                                      'NONFUNCTIONAL', 'REF_CHANNELS')})
PHENO_MARKERS = LINEAGE_MARKERS + STATE_MARKERS
CORRECTED_CSV = OUT + 'cells_corrected.csv'
RES = OUT                                                   # results go into the sample folder
FIG_DIR = output_dir + 'figures/'                           # one figure folder per sample (03a, 03b, 03c)
os.makedirs(FIG_DIR, exist_ok=True)
PARAM_LOG = json.load(open(OUT + 'parameters.json')) if os.path.exists(OUT + 'parameters.json') else {}
PARAM_LOG.update(filtering='SPACEc 03.1 tutorial', pixel_size_um=PIXEL_SIZE_UM)

def finish(name, title=None, fig=None, dpi=300, tight=True):
    """Add '<PLOT_TAG> | [title]' above the figure, save it as figures/<PLOT_TAG>_<name>.png, show it."""
    fig = fig or plt.gcf()
    label = PLOT_TAG + (f'  |  {title}' if title else '')
    if fig.get_constrained_layout():
        fig.suptitle(label, fontsize=14, fontweight='bold')
    else:
        fig.suptitle(label, fontsize=14, fontweight='bold', y=1.02)
    fig.savefig(FIG_DIR + f'{PLOT_TAG}_{name}.png', dpi=dpi, **(dict(bbox_inches='tight') if tight else {}))
    plt.show()

print(f'{PLOT_TAG}: {cfg["n_cells"]:,} cells in the corrected table')
print('lineage markers:', LINEAGE_MARKERS)

## 0.3 Analysis region (QuPath GeoJSON)
Only cells whose centre lies inside your QuPath outline are analysed. With the current 03a this is already the case, so all cells are kept here. For samples processed with an older 03a (whole tissue), the outline is applied now. The area of the region (without empty glass) is used for the cell densities in Part 5.

In [ ]:
def _rings(geom):
    if not geom:
        return
    t, c = geom.get('type'), geom.get('coordinates')
    if t == 'Polygon':
        for k, ring in enumerate(c): yield ring, k == 0
    elif t == 'MultiPolygon':
        for poly in c:
            for k, ring in enumerate(poly): yield ring, k == 0
    elif t in ('LineString', 'MultiLineString'):
        for line in ([c] if t == 'LineString' else c):
            if len(line) >= 3: yield line, True
    elif t == 'GeometryCollection':
        for g in geom.get('geometries', []): yield from _rings(g)

def geojson_rings(path):
    gj = json.load(open(path))
    feats = gj['features'] if isinstance(gj, dict) and 'features' in gj else (gj if isinstance(gj, list) else [gj])
    out = []
    for f in feats:
        geom = f.get('geometry', f) if isinstance(f, dict) else None
        out += [(np.asarray(r, float).reshape(-1, 2), o) for r, o in _rings(geom)]
    assert any(o for _, o in out), f'no area found in {path}'
    return out

df = pd.read_csv(CORRECTED_CSV, index_col='label')
masks = np.load(OUT + 'masks_lowres.npz')
tissue_lr = masks['tissue_lr']
px_mm2 = (BG_DS * PIXEL_SIZE_UM / 1000) ** 2                     # area of one low-res pixel in mm²

REGION_GEOJSON = REGION_GEOJSON or cfg.get('REGION_GEOJSON') or cfg.get('TUMOUR_GEOJSON', '')
assert REGION_GEOJSON, 'no GeoJSON outline: give REGION_GEOJSON (column geojson in samples.csv)'
ROI = 'geojson'
rings = geojson_rings(REGION_GEOJSON)
pts = df[['x', 'y']].to_numpy()
in_roi = np.zeros(len(df), bool); roi_lr = np.zeros(tissue_lr.shape, bool)
for r, outline in rings:                                      # outlines add, holes remove
    inside = MplPath(r).contains_points(pts)
    in_roi = (in_roi | inside) if outline else (in_roi & ~inside)
    rr, cc = draw_polygon(r[:, 1] / BG_DS, r[:, 0] / BG_DS, roi_lr.shape)
    roi_lr[rr, cc] = outline
roi_lr &= tissue_lr
ROI_AREA_MM2 = float(roi_lr.sum() * px_mm2)

fig, ax = plt.subplots(figsize=(10, 10 * H / W))
ax.scatter(df.x[~in_roi], df.y[~in_roi], s=0.3, c='0.85', rasterized=True, linewidths=0)
ax.scatter(df.x[in_roi], df.y[in_roi], s=0.3, c='tab:red', rasterized=True, linewidths=0)
ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off')
ax.set_title(f'analysis region: {in_roi.sum():,} of {len(df):,} cells (red), {ROI_AREA_MM2:.2f} mm²')
finish('0-03_analysis_region_cells', 'Cells in the analysis region (QuPath GeoJSON)', dpi=150)

df = df[in_roi].copy()
PARAM_LOG.update(region_geojson=REGION_GEOJSON, roi_cells=int(len(df)), roi_area_mm2=ROI_AREA_MM2)

---
# Part 4: Filtering, normalisation and clustering (SPACEc)
Cell types are identified by unsupervised clustering. The filtering follows the SPACEc preprocessing tutorial (03.1) step by step:
**DAPI/area filter → z-score normalisation → noise removal → AnnData → Leiden clustering**. Subclustering and annotation follow in 03c.

## 4.0 Filter cells by DAPI intensity and area
As in the SPACEc tutorial, the lowest 1 % of cells by nuclear area and the lowest 1 % by DAPI intensity are identified, and cells at or below either value are removed with `sp.pp.filter_data`. These are mostly segmentation artefacts: fragments of nuclei, or objects without a real nucleus.

In [ ]:
import logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'              # silence TensorFlow messages from importing SPACEc
logging.getLogger('absl').setLevel(logging.ERROR)
import spacec as sp
import scanpy as sc
sc.settings.set_figure_params(dpi=80, facecolor='white')
plt.rc('axes', grid=False)                              # no grid lines in figures

clustering_random_seed = 0

ZSCORE_METHOD = 'zscore'     # SPACEc options: 'zscore', 'double_zscore', 'MinMax', 'ArcSin'

# SPACEc table layout: marker columns first, then the columns to keep unchanged (no capping, as in the tutorial)
df_seg = df[marker_channels].copy()
df_seg[NUCLEUS_CHANNEL] = df[NUCLEUS_CHANNEL]
df_seg['x'], df_seg['y'], df_seg['area'] = df['x'], df['y'], df['area']
df_seg['region_num'] = 0
df_seg['unique_region'] = REGION
df_seg['condition'] = CONDITION
df_seg['label'] = df.index.to_numpy()
LIST_KEEP = [NUCLEUS_CHANNEL, 'x', 'y', 'area', 'region_num', 'unique_region', 'condition', 'label']

one_percent_area = np.percentile(df_seg.area, 1)                 # smallest 1 % of cells by area
one_percent_nuc  = np.percentile(df_seg[NUCLEUS_CHANNEL], 1)     # lowest 1 % of cells by DAPI intensity
print(f'1st percentile: area = {one_percent_area:.1f} px², {NUCLEUS_CHANNEL} = {one_percent_nuc:.1f}')

df_filt = sp.pp.filter_data(
    df_seg,
    nuc_thres=one_percent_nuc,       # remove cells with DAPI intensity below threshold
    size_thres=one_percent_area,     # remove cells with area below threshold
    nuc_marker=NUCLEUS_CHANNEL,
    cell_size='area',
    region_column='region_num',
    color_by='region_num',
    log_scale=False)
plt.close('all')                     # filter_data opens an empty figure when plot=False
print(f'Percentage of cells kept: {100 * len(df_filt) / len(df_seg):.2f} % '
      f'({len(df_filt):,} of {len(df_seg):,}; removed {len(df_seg) - len(df_filt):,})')
PARAM_LOG['dapi_area_filter'] = dict(area_thres=float(one_percent_area), nuc_thres=float(one_percent_nuc),
                                     n_before=int(len(df_seg)), n_after=int(len(df_filt)))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, col, thr, ttl in [(axes[0], 'area', one_percent_area, 'nuclear area (px²)'),
                          (axes[1], NUCLEUS_CHANNEL, one_percent_nuc, f'{NUCLEUS_CHANNEL} intensity')]:
    ax.hist(df_seg[col], bins=100, color='0.6')
    ax.axvline(thr, color='red', lw=1.2, ls='--', label=f'1 % = {thr:.1f}')
    ax.set_title(ttl); ax.legend()
plt.tight_layout(); finish('4-00_dapi_area_filter', 'Filter: lowest 1 % by area and DAPI', dpi=150)

## 4.1 Z-score normalisation
Each marker is z-scored per region with `sp.pp.format(method='zscore')`: `z = (value − mean) / SD`. A z-score of 0 is an average cell, and positive values mean above-average staining. As in the tutorial, the values are not capped before z-scoring.

In [ ]:
ZSCORE_METHOD = 'zscore'     # SPACEc options: 'zscore', 'double_zscore', 'MinMax', 'ArcSin'

# This is to normalize the data per region/tif
dfz = pd.DataFrame()
for region in df_filt.unique_region.unique():
    df_reg = df_filt[df_filt.unique_region == region]
    df_reg_norm = sp.pp.format(data=df_reg, list_out=[], list_keep=LIST_KEEP, method=ZSCORE_METHOD)
    dfz = pd.concat([dfz, df_reg_norm], axis=0)

col_num_last_marker = dfz.columns.get_loc(NUCLEUS_CHANNEL) - 1     # index of the last marker column
print(dfz.shape, '| last marker column:', dfz.columns[col_num_last_marker])

## 4.2 Noise removal (top 1 %)
Cells that are positive (z ≥ 0) in almost all markers (`Count`), or that have a very high summed z-score (`z_sum`), are likely debris or segmentation errors. As in the SPACEc tutorial, `sp.pl.zcount_thres` shows both distributions with the cut-off at the top 1 % of cells (`cut_off=0.01`), and `sp.pp.remove_noise` removes every cell above either cut-off. The cut-offs are taken automatically from the same quantile the plot shows. To type them in by hand, as in the tutorial, set `cutoff_list = [[count, z_sum]]`.

In [ ]:
NOISE_CUTOFF = 0.01          # top 1 % of cells (SPACEc tutorial)
cutoff_list  = None          # <-- EDIT None = the cut-offs shown on the plot, or [[z_count_thres, z_sum_thres]] per region

# This function helps to figure out what the cut-off should be for each region
for region in dfz.unique_region.unique():
    print(region)
    df_reg = dfz[dfz.unique_region == region]
    sp.pl.zcount_thres(dfz=df_reg.copy(), col_num=col_num_last_marker,   # last antibody index
                       cut_off=NOISE_CUTOFF,                              # top 1% of cells
                       count_bin=50)
    finish('4-02_noise_thresholds', 'Noise removal: Count and z-score sum (top 1 %)', dpi=150)

In [ ]:
# This is to remove top 1 % of all cells that are highly expressive for all antibodies
df_nn, removed, thresholds = pd.DataFrame(), [], []
for i, region in enumerate(dfz.unique_region.unique()):
    df_reg = dfz[dfz.unique_region == region]
    if cutoff_list is None:          # same numbers as the "Cut off" values on the plot above
        z = df_reg.iloc[:, :col_num_last_marker + 1]
        thr_count = float(z.ge(0).sum(axis=1).quantile(1 - NOISE_CUTOFF))
        thr_sum   = float(z.sum(axis=1).quantile(1 - NOISE_CUTOFF))
    else:
        thr_count, thr_sum = cutoff_list[i]
    print(f'{region}: z_count_thres = {thr_count:.2f}, z_sum_thres = {thr_sum:.2f}')
    df_reg_nn, cc = sp.pp.remove_noise(df=df_reg,
                                       col_num=col_num_last_marker,   # column index of the last protein feature
                                       z_count_thres=thr_count,       # number obtained from the function above
                                       z_sum_thres=thr_sum)           # number obtained from the function above
    print(df_reg_nn.shape)
    df_nn = pd.concat([df_nn, df_reg_nn], axis=0)
    removed.append(cc); thresholds.append([thr_count, thr_sum])
cc = pd.concat(removed)
df_nn = df_nn.reset_index(drop=True)

#Save the df as a backup. We strongly recommend the Anndata format for further analysis!
df_nn.to_csv(RES + 'df_nn.csv', index=False)
PARAM_LOG['noise_removal'] = dict(cut_off=NOISE_CUTOFF, cutoff_list=cutoff_list, thresholds=thresholds,
                                  n_removed=int(len(cc)))

print(f'\nremoved {len(cc):,} cells; mean z of the removed cells (top 10 markers):')
print(cc.iloc[:, :col_num_last_marker + 1].mean().sort_values(ascending=False).head(10).round(2).to_string())
fig, ax = plt.subplots(figsize=(8, 8 * H / W))
ax.scatter(df_filt.x, df_filt.y, s=0.05, c='0.85', rasterized=True)
ax.scatter(cc.x, cc.y, s=4, c='red')
ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off'); ax.set_title(f'{len(cc):,} cells removed as noise (red)')
finish('4-02_noise_removed_map', 'Cells removed as noise', dpi=150)

## 4.3 AnnData object
`sp.hf.make_anndata` turns the table into an AnnData object: the marker z-scores become `X`, and everything after the last marker column (DAPI, x, y, area, region, condition, cell label) goes into `obs`. The non-functional antibodies are dropped here.

In [ ]:
nonFuncAb_list = NONFUNCTIONAL
missing = [m for m in nonFuncAb_list if m not in df_nn.columns]
assert not missing, f'not in df_nn: {missing}'

adata = sp.hf.make_anndata(df_nn=df_nn, col_sum=col_num_last_marker, nonFuncAb_list=nonFuncAb_list)
adata.obs['label'] = adata.obs['label'].astype(int)
adata.obs_names = adata.obs['label'].astype(str).to_numpy()
adata.write_h5ad(RES + 'adata_nn.h5ad')
adata

## 4.4 Leiden clustering
`sp.tl.clustering` builds a k-nearest-neighbour graph of the cells (`N_NEIGHBORS`) from the **lineage markers**, computes a UMAP embedding, and groups the cells with the Leiden algorithm. `RESOLUTION` controls the number of clusters: higher values give more clusters. State markers (Ki67, HIF1A) are not used for clustering, so proliferating or hypoxic cells do not form separate "cell types".

In [ ]:
CLUSTER_MARKERS = LINEAGE_MARKERS                  # markers used for clustering
PLOT_MARKERS    = LINEAGE_MARKERS + STATE_MARKERS  # markers shown in the plots
RESOLUTION      = 1.0                              # <-- EDIT
N_NEIGHBORS     = 10                               # <-- EDIT
CLUSTER_KEY     = 'leiden_1'

adata = sp.tl.clustering(
    adata,
    clustering='leiden',
    n_neighbors=N_NEIGHBORS,
    resolution=RESOLUTION,
    reclustering=False,
    marker_list=CLUSTER_MARKERS,
    key_added=CLUSTER_KEY,
    seed=clustering_random_seed,
)
adata.obs[CLUSTER_KEY] = adata.obs[CLUSTER_KEY].astype('category')
PARAM_LOG['clustering'] = dict(markers=CLUSTER_MARKERS, resolution=RESOLUTION, n_neighbors=N_NEIGHBORS,
                               seed=clustering_random_seed)
print(adata.obs[CLUSTER_KEY].nunique(), 'clusters')

## 4.5 Cluster inspection
* **UMAP** (2D embedding of the marker profiles): clusters coloured by ID, then each marker's z-score.
* **Dotplot**: dot colour = mean expression, dot size = fraction of cells expressing the marker.
* **Matrixplot**: mean z-score per cluster (red = high, blue = low).
* **Cluster summary**: size, the markers with mean z > 1, and a flag for clusters without any lineage marker (tumour by exclusion), with only weak signal, or with markers from several lineages (possible doublets or spillover).
* **Cluster maps**: where each cluster lies in the tissue. Real cell types form structures (vessels, infiltrates, stroma bands). Clusters that follow rectangular acquisition tiles are technical.

In [ ]:
sc.pl.umap(adata, color=[CLUSTER_KEY], legend_loc='on data', size=2, show=False)
finish('4-05_umap_clusters', 'UMAP: Leiden clusters')

sc.pl.umap(adata, color=PLOT_MARKERS, cmap='RdBu_r', vmin=-2, vcenter=0, vmax=3, ncols=5, size=1, show=False)
finish('4-05_umap_markers', 'UMAP: marker z-scores')

sc.pl.dotplot(adata, PLOT_MARKERS, CLUSTER_KEY, dendrogram=True, show=False)
finish('4-05_dotplot_clusters', 'Marker expression per cluster')

sc.pl.matrixplot(adata, PLOT_MARKERS, CLUSTER_KEY, dendrogram=True, colorbar_title='mean z-score',
                 vmin=-2, vmax=2, cmap='RdBu_r', show=False)
finish('4-05_matrixplot_clusters', 'Mean z-score per cluster')

In [ ]:
WEAK_Z = 2.0    # <-- EDIT clusters whose strongest lineage marker is below this are flagged "weak"

def cluster_summary(adata, key, markers=PLOT_MARKERS, thr=1.0):
    """Size, high markers (mean z > thr) and a plausibility flag per cluster."""
    X = adata.to_df()[markers]; cl = adata.obs[key]
    means = X.groupby(cl, observed=True).mean()
    n = cl.value_counts().reindex(means.index)
    high = means[CLUSTER_MARKERS].apply(lambda r: ', '.join(f'{m} {v:.1f}' for m, v in r[r > thr].sort_values(ascending=False).items()) or '-', axis=1)
    state = means[[m for m in STATE_MARKERS if m in means]].apply(lambda r: ', '.join(f'{m} {v:.1f}' for m, v in r[r > thr].items()) or '-', axis=1)
    n_lin = (means[CLUSTER_MARKERS] > 0.5).sum(axis=1)
    top = means[CLUSTER_MARKERS].max(axis=1)
    flag = np.select([n_lin == 0, top < WEAK_Z, n_lin >= 4],
                     ['no lineage marker -> tumour by exclusion?',
                      f'weak (max z < {WEAK_Z:g}) -> check map',
                      'many markers -> doublet/spillover?'], '')
    return pd.DataFrame({'n_cells': n, 'pct': (100 * n / n.sum()).round(1), f'lineage z>{thr}': high,
                         f'state z>{thr}': state, 'flag': flag})

def cluster_maps(adata, key, name, title, ncol=6):
    """One small tissue map per cluster / cell type (red = cells of that group)."""
    cats = list(adata.obs[key].cat.categories)
    xy = adata.obs[['x', 'y']].to_numpy()
    nr = int(np.ceil(len(cats) / ncol))
    fig, axes = plt.subplots(nr, ncol, figsize=(3 * ncol, 3 * nr * H / W + 0.6), squeeze=False)
    for ax, c in zip(axes.ravel(), cats):
        m = (adata.obs[key] == c).to_numpy()
        ax.scatter(xy[:, 0], xy[:, 1], s=0.05, c='0.9', rasterized=True)
        ax.scatter(xy[m, 0], xy[m, 1], s=0.3, c='crimson', rasterized=True)
        ax.set_title(f'{c} (n={m.sum():,})', fontsize=9); ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off')
    for ax in axes.ravel()[len(cats):]:
        ax.axis('off')
    plt.tight_layout(); finish(name, title, dpi=200)

pd.set_option('display.max_colwidth', None)
summary = cluster_summary(adata, CLUSTER_KEY)
summary.to_csv(FIG_DIR + f'{PLOT_TAG}_4-05_cluster_summary.csv')
summary

In [ ]:
cluster_maps(adata, CLUSTER_KEY, '4-05_cluster_maps', 'Spatial location of each cluster')

## 4.5b Save the clustered data for 03c
* `<sample>_adata_clustered.h5ad`: the clustered data (marker z-scores, UMAP, Leiden clusters).
* `<sample>_cluster_config.json`: clustering settings and region area.
* `<sample>_parameters.json`: all settings so far (03a + 03b), continued by 03c.

In [ ]:
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o

adata.write_h5ad(OUT + 'adata_clustered.h5ad')
cluster_config = dict(REGION_GEOJSON=REGION_GEOJSON, ROI=ROI, ROI_AREA_MM2=ROI_AREA_MM2, PIXEL_SIZE_UM=PIXEL_SIZE_UM,
                      CLUSTER_KEY=CLUSTER_KEY, CLUSTER_MARKERS=CLUSTER_MARKERS, PLOT_MARKERS=PLOT_MARKERS,
                      RESOLUTION=RESOLUTION, N_NEIGHBORS=N_NEIGHBORS, clustering_random_seed=clustering_random_seed,
                      WEAK_Z=WEAK_Z, n_cells=int(adata.n_obs), n_clusters=int(adata.obs[CLUSTER_KEY].nunique()))
json.dump(_jsonable(cluster_config), open(OUT + 'cluster_config.json', 'w'), indent=1)
json.dump(_jsonable(PARAM_LOG), open(OUT + 'parameters.json', 'w'), indent=1)
print('written:', OUT + 'adata_clustered.h5ad', f'({adata.n_obs:,} cells, {cluster_config["n_clusters"]} clusters)')
print('done:', PLOT_TAG, '-> annotate in 03c_annotate')